# Вариант 1. Журнал успеваемости

In [1]:
# ===== ЛР 4.6, вариант 1: журнал успеваемости =====
import hashlib
import os
import shutil
from collections import defaultdict

SIGNATURE = "#JOURNAL"
VERSION = 1
CHECK_PREFIX = "#CHECKSUM "
FORBIDDEN = (";", "|", "\n")


class StorageError(Exception):
    """Базовый класс ошибок хранения журнала."""


class FileFormatError(StorageError):
    """Файл не является журналом успеваемости."""


class VersionError(StorageError):
    """Неподдерживаемая версия формата."""
    def __init__(self, found):
        self.found = found
        super().__init__(f"версия {found} не поддерживается (ожидается {VERSION})")


class ChecksumError(StorageError):
    """Файл повреждён или обрезан."""


class CorruptedRecordError(StorageError):
    """Испорчена отдельная запись."""
    def __init__(self, line_no, reason):
        self.line_no = line_no
        super().__init__(f"строка {line_no}: {reason}")


class InvalidRecordError(StorageError):
    """Запись нельзя сохранить."""
    def __init__(self, record_id, reason):
        self.record_id = record_id
        super().__init__(f"{record_id}: {reason}")


class DuplicateEntryError(StorageError):
    """Повтор пары «студент — дисциплина»."""
    def __init__(self, line_no, first_line, student, discipline):
        self.line_no = line_no
        self.first_line = first_line
        self.student = student
        self.discipline = discipline
        super().__init__(
            f"строка {line_no} повторяет строку {first_line} "
            f"({student}, {discipline})")


def checksum(text):
    """Первые 16 символов SHA-256."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


def atomic_write(name, text):
    """Атомарная запись с резервной копией .bak."""
    tmp = name + ".tmp"
    try:
        with open(tmp, "w", encoding="utf-8") as f:
            f.write(text)
    except BaseException:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise
    if os.path.exists(name):
        shutil.copyfile(name, name + ".bak")
    os.replace(tmp, name)


def validate_journal(journal):
    """Проверяет весь журнал до записи."""
    for student, disciplines in journal.items():
        if not student or any(ch in student for ch in FORBIDDEN):
            raise InvalidRecordError(student, "недопустимое ФИО")
        for discipline, grades in disciplines.items():
            if not discipline or any(ch in discipline for ch in FORBIDDEN):
                raise InvalidRecordError(student, "недопустимая дисциплина")
            if not isinstance(grades, list):
                raise InvalidRecordError(student, "оценки должны быть списком")
            for g in grades:
                if not isinstance(g, int) or not 2 <= g <= 5:
                    raise InvalidRecordError(
                        student, f"оценка {g!r} вне диапазона 2..5")


def record_to_line(student, discipline, grades):
    """Пара «студент — дисциплина» → строка файла."""
    return f"{student};{discipline};" + ",".join(str(g) for g in grades)


def line_to_record(line, line_no):
    """Строка файла → (студент, дисциплина, оценки)."""
    parts = line.split(";")
    if len(parts) != 3:
        raise CorruptedRecordError(line_no, f"полей {len(parts)} вместо 3")
    student, discipline, raw = parts
    if not student or not discipline:
        raise CorruptedRecordError(line_no, "пустое ФИО или дисциплина")
    if raw == "":
        grades = []
    else:
        try:
            grades = [int(x) for x in raw.split(",")]
        except ValueError as e:
            raise CorruptedRecordError(line_no, "нечисловая оценка") from e
    for g in grades:
        if not 2 <= g <= 5:
            raise CorruptedRecordError(line_no, f"оценка «{g}» вне диапазона 2..5")
    return student, discipline, grades


def save_journal(name, journal):
    """Сохраняет журнал атомарно."""
    validate_journal(journal)
    body = f"{SIGNATURE} v{VERSION}\n"
    for student in sorted(journal):
        for discipline in sorted(journal[student]):
            body += record_to_line(student, discipline,
                                   journal[student][discipline]) + "\n"
    atomic_write(name, body + CHECK_PREFIX + checksum(body) + "\n")


def load_journal(name, strict=True):
    """Загружает журнал. Возвращает (журнал, ошибки)."""
    try:
        with open(name, encoding="utf-8") as f:
            text = f.read()
    except UnicodeDecodeError as e:
        raise FileFormatError(f"{name}: не текстовый файл UTF-8") from e

    lines = text.split("\n")
    head = lines[0].split()
    if len(head) != 2 or head[0] != SIGNATURE:
        raise FileFormatError(f"{name}: нет сигнатуры {SIGNATURE}")
    try:
        version = int(head[1].lstrip("v"))
    except ValueError as e:
        raise FileFormatError(f"{name}: неверный номер версии") from e
    if version != VERSION:
        raise VersionError(version)

    pos = text.rfind(CHECK_PREFIX)
    if pos == -1:
        raise ChecksumError(f"{name}: нет контрольной суммы — файл обрезан")
    body = text[:pos]
    stored = text[pos + len(CHECK_PREFIX):].strip()
    if checksum(body) != stored:
        raise ChecksumError(f"{name}: контрольная сумма не совпадает")

    journal = {}
    errors = []
    seen = {}
    for line_no, line in enumerate(body.split("\n")[1:-1], start=2):
        try:
            student, discipline, grades = line_to_record(line, line_no)
            key = (student, discipline)
            if key in seen:
                raise DuplicateEntryError(line_no, seen[key], student, discipline)
            journal.setdefault(student, {})[discipline] = grades
            seen[key] = line_no
        except (CorruptedRecordError, DuplicateEntryError) as e:
            if strict:
                raise
            errors.append(e)
    return journal, errors


def load_with_recovery(name):
    """Загрузка с переходом на .bak."""
    try:
        return load_journal(name)[0], "рабочий файл"
    except StorageError as e:
        print("Рабочий файл повреждён:", e)
        return load_journal(name + ".bak")[0], "резервная копия"


def damaged_copy(src, dst, transform):
    with open(src, encoding="utf-8") as f:
        text = f.read()
    with open(dst, "w", encoding="utf-8") as f:
        f.write(transform(text))


def refresh_checksum(text):
    body = text[:text.rfind(CHECK_PREFIX)]
    return body + CHECK_PREFIX + checksum(body) + "\n"


# ---------- демонстрация ----------

journal = {
    "Иванов И. И.": {"Python": [5, 4, 5], "Экология": [4]},
    "Петров П. П.": {"Python": [3, 4], "Математика": [5]},
    "Сидорова А. А.": {"Python": [5], "Экология": [5, 4]},
    "Кузнецов К. К.": {"Математика": [4, 4], "Python": [3]},
    "Смирнов С. С.": {"Экология": [], "Python": [5, 5]},
    "Орлова О. О.": {"Python": [2, 3], "Математика": [4]},
    "Васильев В. В.": {"Python": [4], "Экология": [3, 4]},
    "Николаева Н. Н.": {"Математика": [5, 5], "Python": [5]},
    "Фёдоров Ф. Ф.": {"Python": [3], "Экология": [2]},
}

save_journal("journal.txt", journal)
loaded, errors = load_journal("journal.txt")
print("Сохранено:", len(journal), "студентов,",
      sum(len(d) for d in journal.values()), "записей")
print("Round-trip:", loaded == journal)

for student, disciplines in loaded.items():
    all_grades = [g for grades in disciplines.values() for g in grades]
    avg = sum(all_grades) / len(all_grades) if all_grades else 0
    print(f"{student}: средний балл {avg:.2f}")
print("Есть оценка 2:",
      [s for s, d in loaded.items() if any(2 in g for g in d.values())])

# сценарии отказа
with open("notes.txt", "w", encoding="utf-8") as f:
    f.write("Это не журнал\n")
with open("image.bin", "wb") as f:
    f.write(bytes([0x89, 0x50, 0x4E, 0x47, 0xFF, 0x00, 0xC3]))

damaged_copy("journal.txt", "bad_version.txt",
             lambda t: t.replace("v1", "v2", 1))
damaged_copy("journal.txt", "bad_symbol.txt",
             lambda t: t.replace("Иванов", "Иваново", 1))
damaged_copy("journal.txt", "bad_cut.txt", lambda t: t[:len(t) // 2])
damaged_copy("journal.txt", "bad_grade.txt",
             lambda t: refresh_checksum(t.replace("[5, 4, 5]", "[6, 4, 5]", 1)
                                        .replace("5,4,5", "6,4,5", 1)))
damaged_copy("journal.txt", "bad_dup.txt",
             lambda t: refresh_checksum(
                 t[:t.rfind(CHECK_PREFIX)]
                 + t.split("\n")[1] + "\n"))

for name in ["нет_файла.txt", "notes.txt", "image.bin", "bad_version.txt",
             "bad_symbol.txt", "bad_cut.txt", "bad_grade.txt", "bad_dup.txt"]:
    try:
        load_journal(name)
    except StorageError as e:
        print(f"{name:16} {type(e).__name__}: {e}")
    except OSError as e:
        print(f"{name:16} {type(e).__name__}: файл недоступен")

loaded_soft, errors_soft = load_journal("bad_grade.txt", strict=False)
print("Мягкий режим, загружено записей:",
      sum(len(d) for d in loaded_soft.values()), "| ошибок:", len(errors_soft))

# сбой записи
shutil.copyfile("journal.txt", "work.txt")
try:
    atomic_write("work.txt", 123)  # TypeError внутри записи
except TypeError as e:
    print("Сбой записи:", e)
print("Рабочий файл цел:", load_journal("work.txt")[0] == journal)
print("Временный файл удалён:", not os.path.exists("work.txt.tmp"))

# восстановление
journal["Иванов И. И."]["Python"].append(5)
save_journal("journal.txt", journal)          # .bak — предыдущая версия
damaged_copy("journal.txt", "journal.txt", lambda t: t[:-30])
restored, source = load_with_recovery("journal.txt")
print(f"Восстановлено студентов: {len(restored)}, источник: {source}")

Сохранено: 9 студентов, 18 записей
Round-trip: True
Васильев В. В.: средний балл 3.67
Иванов И. И.: средний балл 4.50
Кузнецов К. К.: средний балл 3.67
Николаева Н. Н.: средний балл 5.00
Орлова О. О.: средний балл 3.00
Петров П. П.: средний балл 4.00
Сидорова А. А.: средний балл 4.67
Смирнов С. С.: средний балл 5.00
Фёдоров Ф. Ф.: средний балл 2.50
Есть оценка 2: ['Орлова О. О.', 'Фёдоров Ф. Ф.']
нет_файла.txt    FileNotFoundError: файл недоступен
notes.txt        FileFormatError: notes.txt: нет сигнатуры #JOURNAL
image.bin        FileFormatError: image.bin: не текстовый файл UTF-8
bad_version.txt  VersionError: версия 2 не поддерживается (ожидается 1)
bad_symbol.txt   ChecksumError: bad_symbol.txt: контрольная сумма не совпадает
bad_cut.txt      ChecksumError: bad_cut.txt: нет контрольной суммы — файл обрезан
bad_grade.txt    CorruptedRecordError: строка 4: оценка «6» вне диапазона 2..5
Мягкий режим, загружено записей: 17 | ошибок: 1
Сбой записи: write() argument must be str, not int


In [ ]:
import glob
import os
"""Deletes all .txt files in the current directory."""
for file in glob.glob("*.txt"):
    try:
        os.remove(file)
        print(f"Deleted: {file}")
    except OSError as e:
        print(f"Error deleting {file}: {e}")